# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR<sup>2</sup> mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review all available record sets and fields, referencing their `@id`s.

We'll list all record sets (`@id` and name) and their fields (by `@id` and name) in the dataset.

In [ ]:
# List all record sets and their fields by @id for easy reference
record_sets = [rs for rs in metadata.record_sets]
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print("  Fields:")
    for f in rs.fields:
        print(f"    Field @id: {f.id}, Name: {f.name}")
    print("-")

## 3. Data Extraction
Load data from each record set using their `@id`. Store each as a DataFrame using the record set `@id` as a key.

We'll extract the first record from each record set and show the available columns.

In [ ]:
# Extract all record sets' records into DataFrames for further analysis.
dataframes = {}
for rs in record_sets:
    # rs.id is the record set's @id
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded records for RecordSet @id={rs.id} - Rows: {len(df)}, Columns: {df.columns.tolist()}")
    if not df.empty:
        print(df.head(2))
    print('-'*60)

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filtering records, normalizing a numeric field, and optionally grouping by a categorical variable. We'll pick a representative numeric field and a group field from the main record set.

### Define the main record set and fields by their `@id`:
- Use the first record set for demonstration purposes.

In [ ]:
# Use the first record set as 'main' for EDA
main_rs = record_sets[0]
main_rs_id = main_rs.id
df = dataframes[main_rs_id]
print(f"Using RecordSet @id: {main_rs_id}")

# Choose numeric and group fields by inspecting the available fields
numeric_field = None
group_field = None
for f in main_rs.fields:
    # Heuristically select a numeric field (e.g., integer or float)
    if getattr(f, 'data_type', None) in {'Float', 'Integer', 'Number'} and numeric_field is None:
        numeric_field = f.id
    # Heuristically select a group field
    if getattr(f, 'data_type', None) == 'Text' and group_field is None:
        group_field = f.id

print(f"Numeric field: {numeric_field}")
print(f"Group field: {group_field}")

# EDA steps
if (numeric_field is not None) and (numeric_field in df.columns):
    threshold = df[numeric_field].astype(float).mean()
    filtered_df = df[df[numeric_field].astype(float) > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} records")
    # Normalize numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
    ) / filtered_df[numeric_field].astype(float).std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Group by group_field if present
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped by {group_field}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize the distribution of the numeric field and its relationship to the group field.

We'll use matplotlib to plot the distribution if EDA fields were found.

In [ ]:
import matplotlib.pyplot as plt

if (numeric_field is not None) and (numeric_field in df.columns):
    fig, ax = plt.subplots(figsize=(6, 4))
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    df[numeric_field].plot.hist(bins=30, alpha=0.7, ax=ax)
    ax.set_title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    # Boxplot by group field if present
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8, 4))
        df.boxplot(column=numeric_field, by=group_field, rot=80)
        plt.title(f"{numeric_field} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion
- We successfully loaded the FAIR² mass spectrometry dataset using its Croissant metadata descriptor via `mlcroissant`.
- We identified the main record sets and their fields, loaded data, and demonstrated EDA including filtering, normalization, and grouping using only field `@id`s for precise, reproducible referencing.
- Data distributions and groupwise differences can be readily visualized for deeper insight or hypothesis generation.